# Submission V3 — 234-class multi-label inference

Uses the V3 model (ConvNeXt-Small, v1 mel params, trained on original data + pseudo-labels).
Inference uses v1 mel params (HOP_LENGTH=512, librosa defaults) to match training.

In [ ]:
import json
import warnings
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
from PIL import Image
import torch
from fastai.vision.all import load_learner

warnings.filterwarnings('ignore', category=UserWarning, module='fastai')

In [ ]:
import kagglehub
path = kagglehub.competition_download('birdclef-2026')
print('Path to competition files:', path)

In [ ]:
MODEL_PATH = '/kaggle/input/models/ucheozoemena/bird-clef-classifier/pytorch/multilabel_234_v3/3/model_multilabel_234.pkl'
VOCAB_PATH = '/kaggle/input/models/ucheozoemena/bird-clef-classifier/pytorch/multilabel_234_v3/3/vocab.json'
TEST_DIR = Path(path) / 'test_soundscapes'
TARGET_SIZE = (224, 224)
CLIP_DURATION = 5
SAMPLE_RATE = 32000
BATCH_SIZE = 64

In [ ]:
sample_sub = pd.read_csv(Path(path) / 'sample_submission.csv')
all_species = [c for c in sample_sub.columns if c != 'row_id']
assert len(all_species) == 234, f'expected 234, got {len(all_species)}'

learn = load_learner(MODEL_PATH, cpu=True)
model_vocab = list(learn.dls.vocab)

if Path(VOCAB_PATH).exists():
    saved_vocab = json.load(open(VOCAB_PATH))
    assert saved_vocab == model_vocab, 'vocab.json disagrees with learner.dls.vocab'

assert model_vocab == all_species, (
    'Model vocab does not match sample_submission column order. '
    f'len(model_vocab)={len(model_vocab)}, len(all_species)={len(all_species)}; '
    f'first mismatch at {next((i for i, (a, b) in enumerate(zip(model_vocab, all_species)) if a != b), None)}'
)
print('Vocab order matches sample_submission for all 234 classes.')

In [ ]:
from torchvision import transforms as T

# v1 mel params — must match training spectrogram generation
HOP_LENGTH = 512
STRIDE_DURATION = 2.5
FRAMES_PER_CLIP = int(CLIP_DURATION * SAMPLE_RATE / HOP_LENGTH)
stride_samples = int(STRIDE_DURATION * SAMPLE_RATE)
stride_frames = int(STRIDE_DURATION * SAMPLE_RATE / HOP_LENGTH)

def window_to_img(window):
    s_min, s_max = float(window.min()), float(window.max())
    if s_max == s_min:
        s_norm = np.zeros_like(window, dtype=np.uint8)
    else:
        s_norm = ((window - s_min) / (s_max - s_min) * 255).astype(np.uint8)
    return Image.fromarray(s_norm).resize(TARGET_SIZE).convert('RGB')

tfm = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

learn.model.eval()
device = next(learn.model.parameters()).device

def run_batch_tta(imgs):
    """Run inference with TTA: average original + time-flipped predictions."""
    preds = []
    for i in range(0, len(imgs), BATCH_SIZE):
        batch_imgs = imgs[i:i + BATCH_SIZE]
        orig    = torch.stack([tfm(img) for img in batch_imgs]).to(device)
        flipped = torch.stack([tfm(img.transpose(Image.FLIP_LEFT_RIGHT)) for img in batch_imgs]).to(device)
        with torch.no_grad():
            p_orig = torch.sigmoid(learn.model(orig)).cpu().numpy()
            p_flip = torch.sigmoid(learn.model(flipped)).cpu().numpy()
        preds.append((p_orig + p_flip) / 2.0)
    return np.vstack(preds)

test_files = sorted(TEST_DIR.glob('*.ogg'))
if not test_files:
    fallback_dir = Path(path) / 'train_soundscapes'
    test_files = sorted(fallback_dir.glob('*.ogg'))[:3]
    print(f'[dry-run] test_soundscapes empty, using {len(test_files)} train_soundscapes files')
else:
    print(f'Found {len(test_files)} test soundscape files')

clip_length = CLIP_DURATION * SAMPLE_RATE
row_ids = []
all_preds = []

for soundscape in test_files:
    samples, _ = librosa.load(soundscape, sr=SAMPLE_RATE)

    S_db = librosa.power_to_db(
        librosa.feature.melspectrogram(y=samples, sr=SAMPLE_RATE, hop_length=HOP_LENGTH),
        ref=np.max,
    )

    overlap_imgs = []
    k = 0
    while k * stride_samples + clip_length <= len(samples):
        window = S_db[:, k * stride_frames : k * stride_frames + FRAMES_PER_CLIP]
        overlap_imgs.append(window_to_img(window))
        k += 1
    n_overlapping = len(overlap_imgs)

    overlap_preds = run_batch_tta(overlap_imgs)

    n_submission = len(samples) // clip_length
    for j in range(n_submission):
        k_lo = max(0, 2 * j - 1)
        k_hi = min(n_overlapping - 1, 2 * j + 1)
        all_preds.append(overlap_preds[k_lo:k_hi + 1].max(axis=0))
        row_ids.append(f'{soundscape.stem}_{(j + 1) * CLIP_DURATION}')

preds_np = np.vstack(all_preds)
assert preds_np.shape == (len(row_ids), 234), preds_np.shape
print(f'Inference complete: {preds_np.shape[0]} clips across {len(test_files)} files')

In [ ]:
submission = pd.DataFrame(preds_np, columns=all_species)
submission.insert(0, 'row_id', row_ids)
assert list(submission.columns) == ['row_id'] + all_species
submission.to_csv('submission.csv', index=False)
print(f'Done. {len(submission)} rows written.')
submission.head()